##### HIGH LEVEL IMPLEMENTATION OF THE PAPER 'Neural Machine Translation by Jointly Learning to Align and Translate (Bahdanau et al., 2014) ' USING PYTORCH

here is the link to the paper: https://arxiv.org/abs/1409.0473

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import re
from torch.nn.utils.rnn import pad_sequence, pad_packed_sequence, pack_padded_sequence
import torch.nn as nn
import torch.nn.functional as F
import random
import time
import math


#### if you are using google colab , uncomment the next section to download the data

In [2]:
# if you are using google colab , uncomment this section to download the data
# import urllib.request
# import zipfile
# import os

# # Download a small, clean dataset of English-German pairs
# url = "https://www.manythings.org/anki/deu-eng.zip"
# zip_path = "deu-eng.zip"

# # Create a request with a User-Agent header to avoid HTTP Error 406
# req = urllib.request.Request(
#     url,
#     headers={
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
#     },
# )

# # Open the URL and retrieve the data
# with urllib.request.urlopen(req) as response:
#     with open(zip_path, 'wb') as out_file:
#         out_file.write(response.read())

# # Extract it
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(".")

In [3]:
with open("../deu.txt", "r", encoding="utf-8") as f:
    lines = f.read().strip().split("\n")
# Keep a subset of 10,000 short sentences to save memory and train fast
pairs = []
for line in lines[17:20]:
    # The file is tab-separated: English \t German \t Attribution
    parts = line.split("\n")[0].split("\t")
    pairs.append((parts[0], parts[1]))

print(f"Sample pair: {pairs[:3]}")

Sample pair: [('Wait.', 'Warte.'), ('Begin.', 'Fang an.'), ('Do it.', 'Mache es!')]


In [4]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.num_words = 4

    def add_sentence(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.num_words
                self.idx2word[self.num_words] = word
                self.num_words += 1

    def numericalize(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        return [
            self.word2idx.get(word, self.word2idx["<UNK>"]) for word in sentence.split()
        ]


# Initialize separate vocabularies for Source and Target languages
eng_vocab = Vocabulary()
deu_vocab = Vocabulary()

for eng, deu in pairs:
    eng_vocab.add_sentence(eng)
    deu_vocab.add_sentence(deu)

In [5]:
type(pairs[0])

tuple

In [6]:
print(pairs)

[('Wait.', 'Warte.'), ('Begin.', 'Fang an.'), ('Do it.', 'Mache es!')]


In [7]:
print(eng_vocab)

In [8]:
print(eng_vocab.word2idx)
print(eng_vocab.idx2word)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'wait': 4, 'begin': 5, 'do': 6, 'it': 7}
{0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>', 4: 'wait', 5: 'begin', 6: 'do', 7: 'it'}


In [9]:
print(deu_vocab.word2idx)
print(deu_vocab.idx2word)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'warte': 4, 'fang': 5, 'an': 6, 'mache': 7, 'es': 8}
{0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>', 4: 'warte', 5: 'fang', 6: 'an', 7: 'mache', 8: 'es'}


In [10]:
print(eng_vocab.numericalize("hi, go out "))

[3, 3, 3]


In [11]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        src_sent, trg_sent = self.pairs[idx]
        # Numericalize text strings
        src_tokens = self.src_vocab.numericalize(src_sent)
        trg_tokens = self.trg_vocab.numericalize(trg_sent)

        # src_tokens = src_tokens[::-1]
        src_tensor = torch.tensor(
            src_tokens + [self.src_vocab.word2idx["<EOS>"]], dtype=torch.long
        )

        trg_tensor = torch.tensor(
            [self.trg_vocab.word2idx["<SOS>"]]
            + trg_tokens
            + [self.trg_vocab.word2idx["<EOS>"]],
            dtype=torch.long,
        )
        return src_tensor, trg_tensor


In [12]:
# Collate function to dynamically pad batches to equivalent lengths
def collate_fn(batch):

    src_batch, trg_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return src_batch, trg_batch


In [13]:
dataset = TranslationDataset(pairs, eng_vocab, deu_vocab)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [14]:
ss = torch.tensor([19, 99, 483, 132, 1275, 3], dtype=torch.long)
print(ss, ss.shape)

tensor([  19,   99,  483,  132, 1275,    3]) torch.Size([6])


In [15]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        print(">>>>>>>>>>>>>>>>>>>>>>>>starting encoder initialization")
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(input_dim, emb_dim)
        print("inputs: ")
        print("hidden_dim: ", hidden_dim)
        print("embedding_dim: ", emb_dim)
        print("embedding: ", self.embedding)
        # Bidirectional, possibly multi-layer LSTM encoder (paper uses a BiRNN, Sec 3.2)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)

        print("LSTM: ")
        print(self.rnn)
        # The decoder is unidirectional, so we need to turn the encoder's final
        # forward+backward states into a single initial state for the decoder.
        # This is the s_0 = tanh(Ws * h_1_backward) idea from Appendix A.2.2,
        # generalized here to combine both directions and, if n_layers > 1,
        # done independently per layer.
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)
        print("fc_hidden linear layer: ", self.fc_hidden)
        print("fc_cell linear layer: ", self.fc_cell)
        print("--------------------- end Encoder initialization ---------------")

    def forward(self, src):
        print(">>>>>>>>>>>>>>>>>>>>>>>>> encoder foward starting")
        print("source shape: ", src.shape)
        print("source:", src)
        # src: (batch, src_len)
        embedded = self.embedding(src)  # (batch, src_len, emb_dim)
        print("embedding shape: ", embedded.shape)
        # print("embedding: ", embedded)

        lengths = (src != 0).sum(dim=1).cpu()  # Shape: (batch)
        packed_embedded = pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        print("lengths for packed sequences: ", lengths)
        # encoder_outputs are the annotations h_j from Eq. 7 (concatenated fwd/bwd
        # states at every source position) -> (batch, src_len, hidden_dim * 2)
        packed_outputs, (hidden, cell) = self.rnn(packed_embedded)
        print("hidden shape: ", hidden.shape)
        print("cell shape: ", cell.shape)

        encoder_outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        print("encoder outputs: ", encoder_outputs.shape)
        # # concatenate the two directions
        # ordered as [layer0_fwd, layer0_bwd, layer1_fwd, layer1_bwd,]
        h_cat = torch.cat((hidden[0], hidden[1]), dim=1)
        c_cat = torch.cat((hidden[0], hidden[1]), dim=1)
        print("h_cat shape: ", h_cat.shape)
        print("c_cat shape: ", h_cat.shape)

        hidden = torch.tanh(self.fc_hidden(h_cat))
        cell = torch.tanh(self.fc_cell(c_cat))
        print("hidden shape: ", hidden.shape)
        print("cell shape: ", cell.shape)

        hidden = hidden.unsqueeze(0)
        cell = cell.unsqueeze(0)
        print("hidden unsqueezed: ", hidden.shape)
        print("cell unsqueezed: ", cell.shape)

        print("--------------------- end Encoder forward ---------------")

        return encoder_outputs, hidden, cell


In [16]:
# Bahdanau additive attention, Eqs. (5)-(6) and Appendix A.1.2:
# e_ij = v_a^T tanh(W_a s_(i-1) + U_a h_j)
class AlignmentModel(nn.Module):
    def __init__(self, decoder_hidden_dim, encoder_output_dim, attention_neurons_dim):
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>> Alignment initialziainog starting")
        super().__init__()
        self.Wa = nn.Linear(decoder_hidden_dim, attention_neurons_dim)
        self.Ua = nn.Linear(encoder_output_dim, attention_neurons_dim)
        self.Va = nn.Linear(attention_neurons_dim, 1)
        print("decoder_hidden_dim: ", decoder_hidden_dim)
        print("encoder_output_dim: ", encoder_output_dim)
        print("attention_neurons_dim: ", attention_neurons_dim)
        print("Wa : ", self.Wa)
        print("Ua :", self.Ua)
        print("Va: ", self.Va)

        print("--------------------- end AlignmentModel initialization ---------------")

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>  Alignment forward starting")
        print("inputs: ")
        print("decoder_hidden shape: ", decoder_hidden.shape)
        print("encoder_outputs shape: ", encoder_outputs.shape)

        # decoder_hidden:  (batch, decoder_hidden_dim)          -- this is s_(i-1)
        # encoder_outputs: (batch, src_len, encoder_output_dim) -- these are h_1..h_Tx
        U_a = self.Ua(encoder_outputs)  # (batch, src_len, attn_dim)
        W_a = self.Wa(decoder_hidden).unsqueeze(1)  # (batch, 1, attn_dim)
        energy = torch.tanh(W_a + U_a)  # (batch, src_len, attn_dim)
        scores = self.Va(energy)  # (batch, src_len, 1) -- e_ij
        scores = scores.squeeze(2)  # (batch, src_len)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            print("mask shape: ", mask.shape)
            print("mask: ", mask)
            print("scores: ", scores)

        print("U_a shape: ", U_a.shape)
        print("W_a shape: ", W_a.shape)
        print("energy: ", energy.shape)
        print("scores: ", scores.shape)
        print("scores:", scores)
        attention_weights = F.softmax(scores, dim=1)  # alpha_ij, softmax over src_len
        print("attention weights shape: ", attention_weights.shape)
        print("attention weights: ", attention_weights)
        print("Encoder outputs: ", encoder_outputs.shape)

        attention_weights = attention_weights.unsqueeze(1)
        print("attention weights unsqueezed: ", attention_weights.shape)
        # c_i = sum_j alpha_ij * h_j  (Eq. 5)
        context_vector = torch.bmm(
            attention_weights, encoder_outputs
        )  # (batch, 1, encoder_output_dim)
        print("context vector shape:", context_vector.shape)
        print("--------------------- end AlignmentModel forward ---------------")

        return context_vector, attention_weights


In [17]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        emb_dim,
        encoder_output_dim,
        decoder_hidden_dim,
        attention_neurons_dim,
    ):
        super().__init__()
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Decoder initialization")
        print("Inputs")
        print("output_dim: ", output_dim)
        print("emb_dim: ", emb_dim)
        print("encoder_output_dim: ", encoder_output_dim)
        print("decoder_hidden_dim: ", decoder_hidden_dim)
        print("attention_neurons_dim: ", attention_neurons_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = AlignmentModel(
            decoder_hidden_dim, encoder_output_dim, attention_neurons_dim
        )
        print("embedding: ", self.embedding)
        print("attention: ", self.attention)
        lstm_input_dim = emb_dim + encoder_output_dim
        self.rnn = nn.LSTM(lstm_input_dim, decoder_hidden_dim, batch_first=True)
        print("lstm_input_dim: ", lstm_input_dim)
        print("LSTM: ", self.rnn)
        self.maxout_linear = nn.Linear(
            decoder_hidden_dim + emb_dim + encoder_output_dim,
            decoder_hidden_dim * 2,
        )
        self.fc_out = nn.Linear(decoder_hidden_dim, output_dim)
        print("maxout linear: ", self.maxout_linear)
        print("fc_out: ", self.fc_out)
        print("--------------------- end decoder initialization ---------------")

    def forward(self, encoder_outputs, input_token, hidden, cell, mask=None):
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Decoder forward starting")

        print("encoder outputs shape: ", encoder_outputs.shape)
        print("input_token shape: ", input_token.shape)
        print("input_token: ", input_token)
        print("hidden shape: ", hidden.shape)
        print("cell shape: ", cell.shape)
        # input_token: (batch,) -> (batch, 1)
        input_token = input_token.unsqueeze(1)
        embedded = self.embedding(input_token)  # (batch, 1, emb_dim)
        print("input_token shape unsqueezed: ", input_token.shape)
        print("input_token  unsqueezed: ", input_token)
        print("embedded shape: ", embedded.shape)

        # Attention query is s_(i-1), the top decoder layer's previous hidden state
        context_vector, attention_weights = self.attention(
            hidden[-1], encoder_outputs, mask=mask
        )
        print("-----decoder continues")
        print("context_vector shape: ", context_vector.shape)
        print("attention weigts shape: ", attention_weights.shape)

        lstm_input = torch.cat((embedded, context_vector), dim=2)
        all_output, (hidden, cell) = self.rnn(lstm_input, (hidden, cell))
        print("lstm_input shape: ", lstm_input.shape)
        # print("all_output shape: ", all_output.shape)
        print("hidden shape: ", hidden.shape)
        print("cell shape: ", cell.shape)

        context_vector = context_vector.squeeze(1)
        embedded = embedded.squeeze(1)
        # top_hidden = hidden[-1]  # (batch, decoder_hidden_dim), this is s_i
        print("context vector: ", context_vector.shape)
        print("embedded : ", embedded.shape)
        # print("top_hidden: ", top_hidden.shape)

        # Deep output / maxout layer (Appendix A.1.1, A.2.2)
        combined_features = torch.cat((hidden[-1], embedded, context_vector), dim=1)
        linear_project = self.maxout_linear(combined_features)
        print("combined_feature: ", combined_features.shape)
        print("linear_project: ", linear_project.shape)

        t1, t2 = torch.chunk(linear_project, 2, dim=1)
        maxout_output = torch.max(t1, t2)  # [batch_size, decoder_hidden_dim]
        print("t1: ", t1.shape)
        print("t2: ", t2.shape)
        print("max_output shape: ", maxout_output.shape)

        prediction = self.fc_out(maxout_output)  # [batch_size, output_dim]
        print("prediction: ", prediction.shape)
        print("--------------------- end decoder initialization ---------------")

        return prediction, hidden, cell, attention_weights


In [18]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>> seq2seq initialization ")
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        print("inputs: ")
        print("encoder: ", encoder)
        print("decoder: ", decoder)
        print("device: ", device)
        print("--------------------- end seq2seq initialization ---------------")

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>> seq2seq forward starting ")
        print("src shape: ", src.shape)
        print("src: ", src)
        print("trg shape: ", trg.shape)
        print("trg : ", trg)

        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        print("batch_size: ", batch_size)
        print("trg len: ", trg_len)
        print("trg_vocab_size: ", trg_vocab_size)
        print("outputs: ", outputs.shape)

        mask = (src != 0).to(self.device)
        print("mask shape: ", mask.shape)
        print("mask: ", mask)
        # encoder_outputs = annotations h_j used by attention every decoding step
        # hidden/cell = projected initial decoder state s_0
        encoder_outputs, hidden, cell = self.encoder(src)
        print("---seq2seq continue")
        print("encoder outputs: ", encoder_outputs.shape)
        print("hidden shape: ", hidden.shape)
        print("cell shape: ", cell.shape)

        input_token = trg[:, 0]  # <SOS>
        print("input token: ", input_token)
        for t in range(1, trg_len):
            prediction, hidden, cell, attention_weights = self.decoder(
                encoder_outputs, input_token, hidden, cell, mask=mask
            )
            print("prediction: ", prediction.shape)
            print("hidden shape: ", hidden.shape)
            print("cell shape: ", cell.shape)

            outputs[:, t, :] = prediction

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            input_token = trg[:, t] if teacher_force else top1
        print("--------------------- end seq2seq initialization ---------------")

        return outputs


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


INPUT_DIM = eng_vocab.num_words
OUTPUT_DIM = deu_vocab.num_words
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HIDDEN_DIM = 256  # decoder hidden size, and per-direction encoder hidden size
ATTN_DIM = 64  # size of the alignment model's hidden layer (n' in the paper)

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM)
# encoder is bidirectional, so its per-position output is HIDDEN_DIM * 2 wide
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM * 2, HIDDEN_DIM, ATTN_DIM)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)  # Ignore <PAD> tokens in loss calculation

print(f"Model initialized successfully. Device: {device}")


>>>>>>>>>>>>>>>>>>>>>>>>starting encoder initialization
inputs: 
hidden_dim:  256
embedding_dim:  128
embedding:  Embedding(8, 128)
LSTM: 
LSTM(128, 256, batch_first=True, bidirectional=True)
fc_hidden linear layer:  Linear(in_features=512, out_features=256, bias=True)
fc_cell linear layer:  Linear(in_features=512, out_features=256, bias=True)
--------------------- end Encoder initialization ---------------
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Decoder initialization
Inputs
output_dim:  9
emb_dim:  128
encoder_output_dim:  512
decoder_hidden_dim:  256
attention_neurons_dim:  64
>>>>>>>>>>>>>>>>>>>>>>>>>>>> Alignment initialziainog starting
decoder_hidden_dim:  256
encoder_output_dim:  512
attention_neurons_dim:  64
Wa :  Linear(in_features=256, out_features=64, bias=True)
Ua : Linear(in_features=512, out_features=64, bias=True)
Va:  Linear(in_features=64, out_features=1, bias=True)
--------------------- end AlignmentModel initialization ---------------
embedding:  Embedding(9, 128)
atten

In [20]:
def train(model, dataloader, optimizer, criterion, clip):
    print(">>>>>>>>>>> starting training looopp. ")
    print("parameters")
    print("model: ", model)
    print("dataloader: ", dataloader)
    print("optimizer: ", optimizer)
    print("criterion: ", criterion)
    print("clip: ", clip)

    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(dataloader):
        print("training loop")
        print("index: ", i)
        print("src: ", src)
        print("trg: ", trg)
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()
        print("optimizer: ", optimizer)
        # Forward pass: pass data through the model layout
        output = model(src, trg)
        print("output shape: ", output.shape)
        # trg = [batch size, trg len]
        # output = [batch size, trg len, output dim]
        print("trg: ", trg.shape)
        output_dim = output.shape[-1]
        print("output_dim", output_dim)
        # Reshape output and target tensors to compute cross-entropy loss
        output = output[:, 1:].reshape(-1, output_dim)
        print("output shape: ", output.shape)
        # print("output: ", output)
        trg = trg[:, 1:].reshape(-1)
        print("trg shape: ", trg.shape)
        print("trg: ", trg)

        loss = criterion(output, trg)
        print("loss : ", loss)
        loss.backward()

        # Gradient clipping - a standard trick for training stable LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


# Run a quick training cycle test
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    start_time = time.time()
    train_loss = train(model, dataloader, optimizer, criterion, CLIP)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )

>>>>>>>>>>> starting training looopp. 
parameters
model:  Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(8, 128)
    (rnn): LSTM(128, 256, batch_first=True, bidirectional=True)
    (fc_hidden): Linear(in_features=512, out_features=256, bias=True)
    (fc_cell): Linear(in_features=512, out_features=256, bias=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(9, 128)
    (attention): AlignmentModel(
      (Wa): Linear(in_features=256, out_features=64, bias=True)
      (Ua): Linear(in_features=512, out_features=64, bias=True)
      (Va): Linear(in_features=64, out_features=1, bias=True)
    )
    (rnn): LSTM(640, 256, batch_first=True)
    (maxout_linear): Linear(in_features=896, out_features=512, bias=True)
    (fc_out): Linear(in_features=256, out_features=9, bias=True)
  )
)
dataloader:  <torch.utils.data.dataloader.DataLoader object at 0x112899520>
optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiabl

In [21]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()  # Turn off dropout / batch norm layers

    # 1. Preprocess the input sentence exactly like training
    tokens = src_vocab.numericalize(sentence)
    tokens = tokens + [src_vocab.word2idx["<EOS>"]]

    # 2. Convert to tensor and add a batch dimension (shape: [1, seq_len])
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    # 3. Pass through Encoder to get annotations + initial decoder state
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

    # 4. Initialize the loop with <SOS>
    trg_indexes = [trg_vocab.word2idx["<SOS>"]]
    attentions = []

    # 5. Decode step-by-step
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            prediction, hidden, cell, attn = model.decoder(
                encoder_outputs, trg_tensor, hidden, cell
            )

        attentions.append(attn.squeeze().cpu())

        # Get the model's highest scoring prediction
        pred_token = prediction.argmax(1).item()
        trg_indexes.append(pred_token)

        # Stop if the model predicts the End of Sentence token
        if pred_token == trg_vocab.word2idx["<EOS>"]:
            break

    # 6. Convert integer indices back into actual German words
    trg_tokens = [trg_vocab.idx2word[i] for i in trg_indexes]

    # Return string while cutting out the structural control symbols, plus
    # attentions so you can plot alignment heatmaps like Fig. 3 in the paper
    return " ".join(trg_tokens[1:-1]), attentions


# --- TEST IT OUT ---
example_input = "Hi, go away"
translation = translate_sentence(example_input, eng_vocab, deu_vocab, model, device)
print(f"English: {example_input}")
print(f"German Translation: {translation}")


>>>>>>>>>>>>>>>>>>>>>>>>> encoder foward starting
source shape:  torch.Size([1, 4])
source: tensor([[3, 3, 3, 2]])
embedding shape:  torch.Size([1, 4, 128])
lengths for packed sequences:  tensor([4])
hidden shape:  torch.Size([2, 1, 256])
cell shape:  torch.Size([2, 1, 256])
encoder outputs:  torch.Size([1, 4, 512])
h_cat shape:  torch.Size([1, 512])
c_cat shape:  torch.Size([1, 512])
hidden shape:  torch.Size([1, 256])
cell shape:  torch.Size([1, 256])
hidden unsqueezed:  torch.Size([1, 1, 256])
cell unsqueezed:  torch.Size([1, 1, 256])
--------------------- end Encoder forward ---------------
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Decoder forward starting
encoder outputs shape:  torch.Size([1, 4, 512])
input_token shape:  torch.Size([1])
input_token:  tensor([1])
hidden shape:  torch.Size([1, 1, 256])
cell shape:  torch.Size([1, 1, 256])
input_token shape unsqueezed:  torch.Size([1, 1])
input_token  unsqueezed:  tensor([[1]])
embedded shape:  torch.Size([1, 1, 128])
>>>>>>>>>>>>>>>>>>>>

NameError: name 'torch' is not defined